# Selected Tab Export

Use this one-time notebook in the repo root to:

1. point to your Excel databook
2. list all available sheet names
3. paste the exact tabs you want
4. export the selected tabs to a text file


In [ ]:
from pathlib import Path

import pandas as pd

workbook_path = Path("databook.xlsx")

sheet_names = pd.ExcelFile(workbook_path, engine="openpyxl").sheet_names
print("Available sheets:")
for idx, name in enumerate(sheet_names, start=1):
    print(f"{idx}. {name}")


In [ ]:
selected_tabs = [
    # Paste the exact sheet names you want to export, for example:
    # "Revenue detail",
    # "Cost bridge",
]

selected_tabs


In [ ]:
def _column_letter(column_number):
    result = []
    current = column_number
    while current > 0:
        current, remainder = divmod(current - 1, 26)
        result.append(chr(65 + remainder))
    return "".join(reversed(result))


def _is_empty_value(value):
    if pd.isna(value):
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def _normalize_cell_value(value):
    if _is_empty_value(value):
        return ""
    if hasattr(value, "strftime"):
        try:
            return value.strftime("%Y-%m-%d")
        except Exception:
            pass
    if isinstance(value, float):
        if value.is_integer():
            return str(int(value))
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value).strip()


def _trim_sheet(df, sheet_name):
    if df is None or df.empty:
        raise ValueError(f"Sheet '{sheet_name}' is empty.")

    non_empty_rows = [
        idx for idx in range(len(df.index)) if not df.iloc[idx].map(_is_empty_value).all()
    ]
    non_empty_cols = [
        idx for idx in range(len(df.columns)) if not df.iloc[:, idx].map(_is_empty_value).all()
    ]

    if not non_empty_rows or not non_empty_cols:
        raise ValueError(f"Sheet '{sheet_name}' is empty after trimming blank borders.")

    start_row = min(non_empty_rows)
    end_row = max(non_empty_rows)
    start_col = min(non_empty_cols)
    end_col = max(non_empty_cols)
    trimmed = df.iloc[start_row:end_row + 1, start_col:end_col + 1].copy().fillna("")
    used_range = (
        f"{_column_letter(start_col + 1)}{start_row + 1}:"
        f"{_column_letter(end_col + 1)}{end_row + 1}"
    )
    return trimmed, start_row, used_range


if not selected_tabs:
    raise ValueError("Please fill in at least one sheet name in selected_tabs.")

all_sheets = pd.read_excel(workbook_path, sheet_name=None, header=None, engine="openpyxl")
available_sheets = list(all_sheets.keys())
missing_tabs = [tab for tab in selected_tabs if tab not in available_sheets]
if missing_tabs:
    raise ValueError(
        f"Requested sheet tabs were not found: {', '.join(missing_tabs)}. "
        f"Available sheets: {', '.join(available_sheets)}"
    )

sections = []
for sheet_name in selected_tabs:
    trimmed_df, start_row, used_range = _trim_sheet(all_sheets[sheet_name], sheet_name)
    column_headers = ["ExcelRow"] + [
        f"Col{_column_letter(index + 1)}" for index in range(len(trimmed_df.columns))
    ]
    lines = [
        f"===== SHEET: {sheet_name} =====",
        f"USED_RANGE: {used_range}",
        "| " + " | ".join(column_headers) + " |",
    ]

    for row_offset, (_, row) in enumerate(trimmed_df.iterrows()):
        excel_row = start_row + row_offset + 1
        cells = [_normalize_cell_value(value) for value in row.tolist()]
        lines.append("| " + " | ".join([str(excel_row), *cells]) + " |")

    sections.append("\n".join(lines))

header = [
    f"WORKBOOK: {workbook_path.name}",
    f"SHEETS: {', '.join(selected_tabs)}",
    "",
]
export_text = "\n\n".join(["\n".join(header).rstrip(), *sections]).strip() + "\n"

output_path = workbook_path.with_name(f"{workbook_path.stem}_selected_tabs.txt")
output_path.write_text(export_text, encoding="utf-8")

print(f"Saved to: {output_path}")
print()
print(export_text[:4000])
